In [3]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from scipy.optimize import differential_evolution
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

def round_weights(weights):
    """Round weights according to specifications"""
    w1, w2, w3 = np.round(weights[:3], 2)
    wPL = np.round(weights[3], 7)
    return np.array([w1, w2, w3, wPL])

def prepare_data(data):
    """Enhanced data preparation with feature engineering"""
    # Create copy to avoid modifying original data
    df = data.copy()
    
    # Convert DC to numerical values
    le = LabelEncoder()
    df['DC_encoded'] = le.fit_transform(df['DC'])
    
    # Feature engineering
    df['w1w2'] = df['w1'] * df['w2']
    df['w2w3'] = df['w2'] * df['w3']
    df['w1w3'] = df['w1'] * df['w3']
    df['w_sum'] = df['w1'] + df['w2'] + df['w3']
    df['w_product'] = df['w1'] * df['w2'] * df['w3']
    
    # Standardize wPL separately due to different scale
    scaler = StandardScaler()
    df['wPL_scaled'] = scaler.fit_transform(df[['wPL']])
    
    # Select features
    features = ['w1', 'w2', 'w3', 'wPL_scaled', 'DC_encoded', 
               'w1w2', 'w2w3', 'w1w3', 'w_sum', 'w_product']
    
    X = df[features]
    y = df['range']
    
    return X, y, le, scaler

def train_optimized_model(X, y):
    """Enhanced model training with improved hyperparameter tuning"""
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        
        # Expanded parameter space
        param_space = {
            'max_depth': [3, 4, 5, 6, 7, 8],
            'learning_rate': [0.01, 0.03, 0.05, 0.07, 0.1],
            'n_estimators': [100, 200, 300, 400, 500],
            'min_child_weight': [1, 2, 3, 4, 5],
            'subsample': [0.7, 0.8, 0.9, 1.0],
            'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
            'gamma': [0, 0.1, 0.2, 0.3],
            'reg_alpha': [0, 0.1, 0.5, 1.0],
            'reg_lambda': [0.1, 0.5, 1.0, 2.0]
        }
        
        model = xgb.XGBRegressor(
            objective='reg:squarederror',
            random_state=42,
            early_stopping_rounds=20
        )
        
        random_search = RandomizedSearchCV(
            model, param_space,
            n_iter=50,  # Increased from 20
            cv=5,
            scoring='neg_mean_squared_error',
            random_state=42,
            n_jobs=-1  # Use all available cores
        )
        
        # Fit model with early stopping
        random_search.fit(
            X_train, y_train,
            eval_set=[(X_test, y_test)],
            verbose=False
        )
        
        best_model = random_search.best_estimator_
        
        # Evaluate model
        train_pred = best_model.predict(X_train)
        test_pred = best_model.predict(X_test)
        
        train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
        test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
        train_r2 = r2_score(y_train, train_pred)
        test_r2 = r2_score(y_test, test_pred)
        
        print(f"Train RMSE: {train_rmse:.2f}")
        print(f"Test RMSE: {test_rmse:.2f}")
        print(f"Train R² Score: {train_r2:.4f}")
        print(f"Test R² Score: {test_r2:.4f}")
        
        return best_model
    
    except Exception as e:
        print(f"Error during model training: {str(e)}")
        raise

def objective_function(weights, model, dc_encoded, scaler):
    """Enhanced objective function with feature engineering"""
    weights = round_weights(weights)
    w1, w2, w3, wPL = weights
    
    # Scale wPL
    wPL_scaled = scaler.transform([[wPL]])[0][0]
    
    # Feature engineering
    w1w2 = w1 * w2
    w2w3 = w2 * w3
    w1w3 = w1 * w3
    w_sum = w1 + w2 + w3
    w_product = w1 * w2 * w3
    
    X = np.array([[w1, w2, w3, wPL_scaled, dc_encoded, 
                   w1w2, w2w3, w1w3, w_sum, w_product]])
    
    range_pred = model.predict(X)[0]
    return -range_pred

def find_optimal_weights(model, dc_encoded, scaler):
    """Enhanced optimization with improved settings"""
    bounds = [(0, 1), (0, 1), (0, 1), (0, 0.0001)]
    
    result = differential_evolution(
        objective_function,
        bounds,
        args=(model, dc_encoded, scaler),
        maxiter=200,  # Increased from 100
        popsize=30,   # Increased from 20
        mutation=(0.5, 1.0),
        recombination=0.7,
        seed=42,
        workers=-1    # Use all available cores
    )
    
    optimal_weights = round_weights(result.x)
    return optimal_weights, -result.fun

def optimize_all_dcs(data, model, le, scaler):
    """Find optimal weights for each drive cycle"""
    results = []
    
    for dc in data['DC'].unique():
        print(f"Optimizing for {dc}...")
        dc_encoded = le.transform([dc])[0]
        optimal_weights, max_range = find_optimal_weights(model, dc_encoded, scaler)
        
        results.append({
            'DC': dc,
            'w1': optimal_weights[0],
            'w2': optimal_weights[1],
            'w3': optimal_weights[2],
            'wPL': optimal_weights[3],
            'predicted_range': max_range
        })
    
    return pd.DataFrame(results)

def find_optimal_weights_for_dcs(data):
    """Main execution function with enhanced error handling"""
    try:
        print("Preparing data...")
        X, y, le, scaler = prepare_data(data)
        
        print("Training model...")
        model = train_optimized_model(X, y)
        
        print("\nFinding optimal weights for each drive cycle...")
        results = optimize_all_dcs(data, model, le, scaler)
        
        return results, model, le, scaler
    
    except Exception as e:
        print(f"Error in optimization process: {str(e)}")
        raise

In [4]:
data = pd.read_csv('../Final_DataPoints_With_DC.csv')
results, trained_model, le, scaler = find_optimal_weights_for_dcs(data)

# Display results
print("\nOptimal weights for each drive cycle:")
print(results)

Preparing data...
Training model...
Train RMSE: 5.21
Test RMSE: 14.94
Train R² Score: 0.9541
Test R² Score: 0.6562

Finding optimal weights for each drive cycle...
Optimizing for NEDC_extended...


Process SpawnPoolWorker-56:
Traceback (most recent call last):
  File "/Users/danish/miniforge3/envs/ML/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/danish/miniforge3/envs/ML/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/danish/miniforge3/envs/ML/lib/python3.10/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/Users/danish/miniforge3/envs/ML/lib/python3.10/multiprocessing/queues.py", line 367, in get
    return _ForkingPickler.loads(res)
AttributeError: Can't get attribute 'objective_function' on <module '__main__' (built-in)>
Process SpawnPoolWorker-58:
Traceback (most recent call last):
  File "/Users/danish/miniforge3/envs/ML/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/danish/miniforge3/envs/ML/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._arg

KeyboardInterrupt: 

In [5]:
# savig results to a csv file
results.to_csv('XGBOOST_Optimal_Weights.csv', index=False)